In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression  # Se corrigió a 'L' mayúscula
from sklearn.pipeline import Pipeline

In [ ]:
healthpredict_visits=pd.read_csv(r"C:\Users\DODY DUEÑAS\Documents\M4\Lecturas\HealthPredict\healthpredict_visits.csv")
healthpredict_patients=pd.read_csv(r"C:\Users\DODY DUEÑAS\Documents\M4\Lecturas\HealthPredict\healthpredict_patients.csv")
pacientes2=pd.read_csv(r"C:\Users\DODY DUEÑAS\Documents\M4\Lecturas\HealthPredict\pacientes2.csv")


In [ ]:
df_visits=pd.DataFrame(healthpredict_visits)
df_visits


In [ ]:
df_patients=pd.DataFrame(healthpredict_patients)
df_patients.drop(columns=["name"],inplace=True)
df_patients


In [ ]:
df_patients2=pd.DataFrame(pacientes2)
df_patients2

In [ ]:
df_patients_all = pd.concat(
    [df_patients, pacientes2],
    ignore_index=True
)
df_patients_all

In [ ]:
merged = df_visits.merge(
    df_patients_all,
    on="patient_id",
    how="left"
)

merged

In [ ]:
merged.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Definir las columnas numéricas a analizar
numeric_cols = ["bmi", "glucose", "blood_pressure_sys", "age"]

# Configurar el tamaño de la figura
plt.figure(figsize=(8, 5))

# Crear el boxplot (diagrama de caja)
# Nota: Asegúrate de que 'merged' sea el nombre de tu DataFrame
sns.boxplot(data=merged[numeric_cols], orient="v")

# Añadir títulos y etiquetas
plt.title("Distribución y outliers por variable numérica")
plt.xlabel("Valor")
plt.show()

In [ ]:
cols_outliers = ["bmi", "glucose"]

outlier_stats = {}

for col in numeric_cols:
    # Calcular los cuartiles Q1 (25%) y Q3 (75%)
    Q1 = merged[col].quantile(0.25)
    Q3 = merged[col].quantile(0.75)
    
    # Calcular el Rango Intercuartílico (IQR)
    IQR = Q3 - Q1
    
    # Definir los límites para detectar outliers
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Filtrar los valores que están fuera de los límites
    outliers = merged[(merged[col] < lower) | (merged[col] > upper)][col]
    
    # Calcular el porcentaje de outliers
    pct_outliers = 100 * len(outliers) / len(merged)
    
    # Guardar los resultados en el diccionario
    outlier_stats[col] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_limit": lower,
        "Upper_limit": upper,
        "Outlier_count": len(outliers),
        "Outlier_%": round(pct_outliers, 2)
    }

# Convertir el diccionario a un DataFrame de Pandas y transponerlo (.T)
outlier_df = pd.DataFrame(outlier_stats).T
print(outlier_df)

In [ ]:
# Como el % de outliers es bajo, no se justifica RobustScaler
# ya que disminuye innecesariamente la varianza.
numeric_cols_robust = []
numeric_cols_standard = ["blood_pressure_sys", "age", "bmi", "glucose"]

# Visualizar el DataFrame final antes de procesar
merged

In [ ]:
# Con pandas
# Convertimos la variable categórica 'condition' en variables dummies
# drop_first=True se usa para evitar la trampa de la multicolinealidad
merged = pd.get_dummies(merged, columns=["condition"], drop_first=True)

# Visualizamos el resultado
merged

In [ ]:
# Vamos a seleccionar ahora las columnas de features y la de target.
# Luego las columnas numéricas y las categóricas de las features.

# Creamos una lista de features excluyendo las columnas que no aportan al modelo
# o que son identificadores únicos.
feature_cols = [col for col in merged.columns if col not in ['outcome_risk', 'visit_date', 'visit_id', 'patient_id', 'region']]

# Definimos la columna objetivo
target_cols = 'outcome_risk'

# Visualizamos la lista de características seleccionadas
feature_cols

In [ ]:
# Separar automáticamente columnas numéricas y categóricas
numeric_cols = merged[feature_cols].select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
categoric_cols = merged[feature_cols].select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# Visualizar las columnas categóricas identificadas
categoric_cols

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calcular matriz de correlaciones solo para las variables numéricas + target
# Usamos feature_cols y target_cols definidos anteriormente
corr_matrix = merged[feature_cols + [target_cols]].corr(numeric_only=True)

# Mostrar correlación directa con la variable objetivo en texto
print("Correlación con el target:")
print(corr_matrix[target_cols].sort_values(ascending=False))

# Visualizar heatmap de correlaciones
plt.figure(figsize=(8, 6))

# Filtramos y ordenamos la matriz para que el heatmap sea más fácil de leer
sns.heatmap(corr_matrix[[target_cols]].sort_values(by=target_cols, ascending=False), 
            annot=True, 
            cmap="coolwarm", 
            center=0)

plt.title("Correlación de cada feature con el target")
plt.show()

In [ ]:
##### Ahora podemos analizar con un pairplot la distribución y presencia de outliers en cada categoría
import seaborn as sns
import matplotlib.pyplot as plt

# Definir las columnas numéricas
numeric_cols = ["bmi", "glucose", "blood_pressure_sys", "age"]

# Configurar el tamaño de la figura (opcional, pairplot se ajusta solo)
plt.figure(figsize=(10, 8))

# Crear el pairplot para ver relaciones entre variables
# Nota: pairplot no suele llevar el argumento orient="v" ya que es una matriz de gráficos
sns.pairplot(data=merged[numeric_cols])

# Añadir título general y mostrar
plt.suptitle("Distribución y outliers por variable numérica", y=1.02)
plt.show()

In [ ]:
# Definimos X con las columnas numéricas y categóricas identificadas
X = merged[numeric_cols + categoric_cols]

# Definimos y como el target, asegurándonos de que sea tipo entero
y = merged[target_cols].astype(int)

# Visualizar X para confirmar la estructura
X

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir los datos: 80% para entrenamiento y 20% para prueba
# stratify asegura que la distribución de 'outcome_risk' sea igual en train y test
X_train, X_test, y_train, y_test = train_test_split(
    merged[feature_cols], 
    merged[target_cols], 
    test_size=0.2, 
    random_state=42, 
    stratify=merged[target_cols]
)

In [ ]:
X_test

In [ ]:
# Preparo el pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Pipeline para variables numéricas con muchos outliers (usa mediana y RobustScaler)
numeric_pipe_robust = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

# Pipeline para variables numéricas estándar (usa media y StandardScaler)
numeric_pipe_standard = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Pipeline para variables categóricas (imputa con lo más frecuente y aplica One-Hot)
ohe_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [ ]:
# Realizamos las predicciones con el conjunto de prueba
y_pred = model_pipeline.predict(X_test)

# Importamos las métricas de evaluación
from sklearn.metrics import classification_report, confusion_matrix

# Mostramos el reporte de clasificación (Precisión, Recall, F1-score)
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

# Visualizamos la matriz de confusión
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusión")
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.show()

In [ ]:
# Creamos el pipeline final que incluye el preprocesamiento y el modelo
model_pipeline = Pipeline(steps=[
    ("preprocessor", pre),
    ("classifier", LogisticRegression())
])

# Entrenamos el modelo con los datos de entrenamiento
model_pipeline.fit(X_train, y_train)

# Visualizamos el pipeline para confirmar la estructura
model_pipeline